In [1]:
# Cell 1 — Imports
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import pickle
import math
from torch.utils.data import Dataset, DataLoader

In [2]:
# Cell 2 — Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [3]:
# Cell 3 — Find exact file paths
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/excimusprime/machine-translation-files/eng_embedding_matrix.npy
/kaggle/input/datasets/excimusprime/machine-translation-files/hi_embedding_matrix.npy
/kaggle/input/datasets/excimusprime/machine-translation-files/train_processed.pkl
/kaggle/input/datasets/excimusprime/machine-translation-files/val_processed.pkl
/kaggle/input/datasets/excimusprime/machine-translation-files/eng_vocab.pkl
/kaggle/input/datasets/excimusprime/machine-translation-files/hi_vocab.pkl
/kaggle/input/datasets/excimusprime/machine-translation-files/test_processed.pkl


In [4]:
# Cell 4 — Load all data
DATA_PATH = '/kaggle/input/datasets/excimusprime/machine-translation-files/'

train = pd.read_pickle(DATA_PATH + 'train_processed.pkl')
val   = pd.read_pickle(DATA_PATH + 'val_processed.pkl')
test  = pd.read_pickle(DATA_PATH + 'test_processed.pkl')

with open(DATA_PATH + 'eng_vocab.pkl', 'rb') as f:
    eng_vocab = pickle.load(f)
with open(DATA_PATH + 'hi_vocab.pkl', 'rb') as f:
    hi_vocab = pickle.load(f)

eng_matrix = np.load(DATA_PATH + 'eng_embedding_matrix.npy')
hi_matrix  = np.load(DATA_PATH + 'hi_embedding_matrix.npy')

print(f"Train: {len(train):,}")
print(f"Val:   {len(val):,}")
print(f"Test:  {len(test):,}")
print(f"English vocab: {len(eng_vocab):,} | Matrix: {eng_matrix.shape}")
print(f"Hindi vocab:   {len(hi_vocab):,} | Matrix: {hi_matrix.shape}")

Train: 120,000
Val:   15,000
Test:  15,000
English vocab: 18,671 | Matrix: (18671, 300)
Hindi vocab:   18,715 | Matrix: (18715, 300)


In [5]:
# Cell 5 — Dataset class
class TranslationDataset(Dataset):
    def __init__(self, dataframe):
        self.english = torch.tensor(
            dataframe['eng_padded'].tolist(),
            dtype=torch.long
        )
        self.hindi = torch.tensor(
            dataframe['hi_padded'].tolist(),
            dtype=torch.long
        )

    def __len__(self):
        return len(self.english)

    def __getitem__(self, idx):
        return self.english[idx], self.hindi[idx]

train_dataset = TranslationDataset(train)
val_dataset   = TranslationDataset(val)
test_dataset  = TranslationDataset(test)

print(f"Train dataset: {len(train_dataset):,} pairs")
print(f"Val dataset:   {len(val_dataset):,} pairs")
print(f"Test dataset:  {len(test_dataset):,} pairs")

Train dataset: 120,000 pairs
Val dataset:   15,000 pairs
Test dataset:  15,000 pairs


In [6]:
# Cell 6 — DataLoaders
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Batch size: {BATCH_SIZE}")
print(f"Training batches per epoch:   {len(train_loader):,}")
print(f"Validation batches per epoch: {len(val_loader):,}")

Batch size: 64
Training batches per epoch:   1,875
Validation batches per epoch: 235


In [7]:
# Cell 7 — LSTM Encoder with Self Attention
# KEY DIFFERENCE from RNN:
# LSTM returns (hidden, cell) instead of just hidden

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout, pretrained_embeddings):
        super(Encoder, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight = nn.Parameter(
            torch.tensor(pretrained_embeddings, dtype=torch.float32)
        )
        self.embedding.weight.requires_grad = True

        # LSTM instead of RNN
        self.lstm = nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        # Self Attention layers
        self.attention_q = nn.Linear(hidden_dim, hidden_dim)
        self.attention_k = nn.Linear(hidden_dim, hidden_dim)
        self.attention_v = nn.Linear(hidden_dim, hidden_dim)

        self.dropout = nn.Dropout(dropout)
        self.scale = math.sqrt(hidden_dim)

    def self_attention(self, hidden_states):
        Q = self.attention_q(hidden_states)
        K = self.attention_k(hidden_states)
        V = self.attention_v(hidden_states)

        scores  = torch.bmm(Q, K.transpose(1, 2)) / self.scale
        weights = torch.softmax(scores, dim=-1)
        weights = self.dropout(weights)
        context = torch.bmm(weights, V)
        return context

    def forward(self, src):
        embedded = self.dropout(self.embedding(src))

        # LSTM returns (outputs, (hidden, cell))
        outputs, (hidden, cell) = self.lstm(embedded)

        # Apply self attention
        attended = self.self_attention(outputs)
        outputs  = outputs + attended  # residual connection

        # Return outputs, hidden AND cell
        return outputs, hidden, cell

print("LSTM Encoder defined!")

LSTM Encoder defined!


In [8]:
# Cell 8 — LSTM Decoder
# KEY DIFFERENCE from RNN:
# Decoder also uses LSTM and passes both hidden AND cell state

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout, pretrained_embeddings):
        super(Decoder, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight = nn.Parameter(
            torch.tensor(pretrained_embeddings, dtype=torch.float32)
        )
        self.embedding.weight.requires_grad = True

        # Cross attention layers
        self.attention_q = nn.Linear(hidden_dim, hidden_dim)
        self.attention_k = nn.Linear(hidden_dim, hidden_dim)
        self.attention_v = nn.Linear(hidden_dim, hidden_dim)

        # LSTM instead of RNN
        self.lstm = nn.LSTM(
            input_size=embed_dim + hidden_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )

        self.fc_out  = nn.Linear(hidden_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.scale   = math.sqrt(hidden_dim)

    def attention(self, decoder_hidden, encoder_outputs):
        query = decoder_hidden.permute(1, 0, 2)
        Q = self.attention_q(query)
        K = self.attention_k(encoder_outputs)
        V = self.attention_v(encoder_outputs)
        scores  = torch.bmm(Q, K.transpose(1, 2)) / self.scale
        weights = torch.softmax(scores, dim=-1)
        context = torch.bmm(weights, V)
        return context, weights

    def forward(self, tgt, hidden, cell, encoder_outputs):
        # KEY DIFFERENCE: takes hidden AND cell as separate inputs
        tgt      = tgt.unsqueeze(1)
        embedded = self.dropout(self.embedding(tgt))

        context, weights = self.attention(hidden, encoder_outputs)
        rnn_input = torch.cat([embedded, context], dim=2)

        # LSTM takes (hidden, cell) tuple
        output, (hidden, cell) = self.lstm(rnn_input, (hidden, cell))

        prediction = self.fc_out(output.squeeze(1))

        # Returns hidden AND cell
        return prediction, hidden, cell, weights

print("LSTM Decoder defined!")

LSTM Decoder defined!


In [9]:
# Cell 9 — Seq2Seq
# KEY DIFFERENCE: passes cell state throughout decoding loop

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super(Seq2Seq, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device  = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size     = src.shape[0]
        tgt_len        = tgt.shape[1]
        tgt_vocab_size = self.decoder.fc_out.out_features

        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)

        # Encoder returns outputs, hidden AND cell
        encoder_outputs, hidden, cell = self.encoder(src)

        decoder_input = tgt[:, 0]

        for t in range(1, tgt_len):
            # Decoder takes hidden AND cell, returns both
            prediction, hidden, cell, _ = self.decoder(
                decoder_input, hidden, cell, encoder_outputs
            )
            outputs[:, t, :] = prediction

            use_teacher_forcing = torch.rand(1).item() < teacher_forcing_ratio
            if use_teacher_forcing:
                decoder_input = tgt[:, t]
            else:
                decoder_input = prediction.argmax(1)

        return outputs

print("Seq2Seq defined!")

Seq2Seq defined!


In [10]:
# Cell 10 — Initialize model
EMBED_DIM     = 300
HIDDEN_DIM    = 512
DROPOUT       = 0.3
LEARNING_RATE = 0.0005

encoder = Encoder(
    vocab_size=len(eng_vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    pretrained_embeddings=eng_matrix
).to(device)

decoder = Decoder(
    vocab_size=len(hi_vocab),
    embed_dim=EMBED_DIM,
    hidden_dim=HIDDEN_DIM,
    dropout=DROPOUT,
    pretrained_embeddings=hi_matrix
).to(device)

model = Seq2Seq(encoder, decoder, device).to(device)

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Model initialized!")
print(f"Trainable parameters: {count_parameters(model):,}")

Model initialized!
Trainable parameters: 26,775,251


In [11]:
# Cell 11 — Loss and optimizer
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Loss: CrossEntropyLoss (ignoring PAD)")
print(f"Optimizer: Adam (lr={LEARNING_RATE})")

Loss: CrossEntropyLoss (ignoring PAD)
Optimizer: Adam (lr=0.0005)


In [12]:
# Cell 12 — Training functions
def train_epoch(model, loader, optimizer, criterion, clip=1.0):
    model.train()
    epoch_loss = 0

    for src, tgt in loader:
        src = src.to(device)
        tgt = tgt.to(device)

        optimizer.zero_grad()
        output = model(src, tgt, teacher_forcing_ratio=0.5)

        output = output[:, 1:, :].reshape(-1, output.shape[-1])
        tgt    = tgt[:, 1:].reshape(-1)

        loss = criterion(output, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()

        epoch_loss += loss.item()

    return epoch_loss / len(loader)


def evaluate(model, loader, criterion):
    model.eval()
    epoch_loss = 0

    with torch.no_grad():
        for src, tgt in loader:
            src = src.to(device)
            tgt = tgt.to(device)

            output = model(src, tgt, teacher_forcing_ratio=0.0)

            output = output[:, 1:, :].reshape(-1, output.shape[-1])
            tgt    = tgt[:, 1:].reshape(-1)

            loss = criterion(output, tgt)
            epoch_loss += loss.item()

    return epoch_loss / len(loader)

print("Training functions defined!")

Training functions defined!


In [13]:
# Cell 13 — Train!
N_EPOCHS      = 10
best_val_loss = float('inf')

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

for epoch in range(N_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss   = evaluate(model, val_loader, criterion)

    scheduler.step(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '/kaggle/working/lstm_best_model.pt')
        print(f"Epoch {epoch+1:02d} -> Train: {train_loss:.4f} | Val: {val_loss:.4f} saved")
    else:
        print(f"Epoch {epoch+1:02d} -> Train: {train_loss:.4f} | Val: {val_loss:.4f}")

print(f"\nTraining complete! Best val loss: {best_val_loss:.4f}")

Epoch 01 -> Train: 6.5830 | Val: 6.4983 saved
Epoch 02 -> Train: 6.1084 | Val: 6.1589 saved
Epoch 03 -> Train: 5.4248 | Val: 5.8686 saved
Epoch 04 -> Train: 5.0383 | Val: 5.6773 saved
Epoch 05 -> Train: 4.7696 | Val: 5.5497 saved
Epoch 06 -> Train: 4.5525 | Val: 5.4812 saved
Epoch 07 -> Train: 4.3783 | Val: 5.4647 saved
Epoch 08 -> Train: 4.2289 | Val: 5.4279 saved
Epoch 09 -> Train: 4.1108 | Val: 5.3828 saved
Epoch 10 -> Train: 3.9982 | Val: 5.3821 saved

Training complete! Best val loss: 5.3821


In [14]:
# Cell 14 — Evaluation
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from sklearn.metrics import f1_score

model.load_state_dict(torch.load('/kaggle/working/lstm_best_model.pt'))
model.eval()

idx2hi = {idx: word for word, idx in hi_vocab.items()}
idx2eng = {idx: word for word, idx in eng_vocab.items()}

def indices_to_tokens(indices, idx2word):
    tokens = []
    for idx in indices:
        idx = idx.item()
        if idx == 2:  # EOS
            break
        if idx not in [0, 1]:  # skip PAD and SOS
            tokens.append(idx2word.get(idx, '<UNK>'))
    return tokens

all_predictions = []
all_references  = []
all_pred_flat   = []
all_tgt_flat    = []

print("Evaluating on test set...")
for src, tgt in test_loader:
    src = src.to(device)
    tgt = tgt.to(device)

    with torch.no_grad():
        output = model(src, tgt, teacher_forcing_ratio=0.0)

    pred_indices = output.argmax(dim=2)

    for i in range(pred_indices.shape[0]):
        pred_tokens = indices_to_tokens(pred_indices[i], idx2hi)
        ref_tokens  = indices_to_tokens(tgt[i], idx2hi)

        all_predictions.append(pred_tokens)
        all_references.append([ref_tokens])

        min_len = min(len(pred_tokens), len(ref_tokens))
        if min_len > 0:
            all_pred_flat.extend(pred_tokens[:min_len])
            all_tgt_flat.extend(ref_tokens[:min_len])

print(f"Evaluation done! Total: {len(all_predictions):,}")
print(f"Non-empty predictions: {sum(1 for p in all_predictions if len(p) > 0):,}")

Evaluating on test set...
Evaluation done! Total: 15,000
Non-empty predictions: 15,000


In [15]:
# Cell 15 — Compute Scores
smoothing  = SmoothingFunction().method1
bleu_score = corpus_bleu(all_references, all_predictions,
                          smoothing_function=smoothing)

if all_tgt_flat:
    correct  = sum(p == r for p, r in zip(all_pred_flat, all_tgt_flat))
    accuracy = correct / len(all_tgt_flat)
    unique_labels = list(set(all_tgt_flat))
    f1 = f1_score(all_tgt_flat, all_pred_flat,
                  labels=unique_labels,
                  average='weighted',
                  zero_division=0)
else:
    accuracy = 0
    f1 = 0

print("=" * 50)
print("       LSTM MODEL — TEST SET RESULTS")
print("=" * 50)
print(f"BLEU Score:  {bleu_score:.4f}  ({bleu_score*100:.2f}%)")
print(f"Accuracy:    {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"F1 Score:    {f1:.4f}  ({f1*100:.2f}%)")
print(f"Val Loss:    {best_val_loss:.4f}")
print("=" * 50)

       LSTM MODEL — TEST SET RESULTS
BLEU Score:  0.0616  (6.16%)
Accuracy:    0.1076  (10.76%)
F1 Score:    0.0864  (8.64%)
Val Loss:    5.3821


In [16]:
# Cell 16 — Sample Translations
print("SAMPLE TRANSLATIONS")
print("=" * 60)

src_batch, tgt_batch = next(iter(test_loader))
src_batch = src_batch.to(device)
tgt_batch = tgt_batch.to(device)

with torch.no_grad():
    output = model(src_batch, tgt_batch, teacher_forcing_ratio=0.0)

pred_indices = output.argmax(dim=2)

for i in range(5):
    src_tokens  = indices_to_tokens(src_batch[i], idx2eng)
    ref_tokens  = indices_to_tokens(tgt_batch[i], idx2hi)
    pred_tokens = indices_to_tokens(pred_indices[i], idx2hi)

    print(f"English:   {' '.join(src_tokens[:15])}")
    print(f"Reference: {' '.join(ref_tokens[:15])}")
    print(f"Predicted: {' '.join(pred_tokens[:15])}")
    print("-" * 60)

SAMPLE TRANSLATIONS
English:   a pakistani court has constituted a <UNK> bench to hear an appeal of jailed former
Reference: पाकिस्तान की एक अदालत ने अल <UNK> भ्रष्टाचार मामले में दोषी <UNK> जाने के खिलाफ
Predicted: पाकिस्तान के पूर्व प्रधानमंत्री नवाज शरीफ की पूर्व संध्या पर पूर्व प्रधानमंत्री नवाज शरीफ के
------------------------------------------------------------
English:   the explosion caused panic in the area .
Reference: इस विस्फोट के चलते क्षेत्र में हड़कंप मच गया ।
Predicted: विस्फोट में <UNK> इलाके में दहशत फैल गई ।
------------------------------------------------------------
English:   government approved this
Reference: सरकार ने दी मंजूरी
Predicted: सरकार ने मंजूरी दी
------------------------------------------------------------
English:   india has a population of 130 crore .
Reference: भारत , मन बना चुका है , भारत के करोड़ लोग मन बना चुके हैं
Predicted: भारत में लगभग करोड़ करोड़ रुपये है ।
------------------------------------------------------------
English:   that is why